## WARNING - do not use "Save & Run All" on this notebook

`/kaggle/working/` is scoped to a single session. "Save & Run All" starts a
fresh container with an EMPTY `/kaggle/working/`, so Cell 7's resume logic
will find no checkpoint, conclude 0/150 are done, and re-run the full
150-query Faithfulness pass from scratch - roughly 300 Gemini API calls,
not the 2-query retry you'd expect.

**Safe to use anytime:** Quick Save / "Save Version" without the run-all
option. This snapshots the notebook and its existing cell outputs exactly
as they are, with zero API calls.

**If "Save & Run All" is ever genuinely needed** (e.g. publishing this
notebook for reproducibility, or a from-scratch pipeline verification):
first publish `step8_faithfulness_full150.jsonl` (148/150 succeeded, see
`step8.md` sec 4a.7) as its own Kaggle Dataset, then change Cell 7's
`CKPT_PATH_FULL` to read from that dataset's input path instead of
`/kaggle/working/`, copying it into `/kaggle/working/` at the start of the
cell before the resume check runs. This is NOT done yet - do this first,
or expect a full re-run.

## Cell 0 - Install ragas 0.4.3 (install only, no import yet)

Installs the pinned ragas version and nothing else. It deliberately does NOT
import ragas: earlier attempts in this session left partially-initialised
modules behind, so the import is deferred to Cell 0b after a kernel restart.

Version choice: ragas 0.4.3, matching the API assumed by `step8.md` §4
(`llm_factory(..., provider="google", client=client)`). An earlier attempt
pinned 0.3.9 on the assumption that it avoided a known VertexAI import bug
(GitHub issues #2745 / #2741). Diagnostics in this notebook disproved that:
0.3.9's `ragas/llms/base.py` carries the same two broken imports, so
downgrading buys nothing while costing API compatibility.

After this cell finishes: Run -> Restart Session, then run Cell 0b.
Pip-installed packages survive a kernel restart on Kaggle; only the Python
import state is cleared, which is exactly what we want.

In [ ]:
# Cell 0: install ragas 0.4.3 (no import here, by design)
import subprocess, sys

RAGAS_PIN = "ragas==0.4.3"

print("installing:", RAGAS_PIN)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", RAGAS_PIN],
    check=True,
)

# confirm what is on disk (does NOT import the package)
out = subprocess.run(
    [sys.executable, "-m", "pip", "show", "ragas"],
    capture_output=True, text=True,
)
for line in out.stdout.splitlines():
    if line.startswith(("Name:", "Version:", "Location:")):
        print(line)

print("\nNEXT: Run -> Restart Session, then run Cell 0b.")
print("Do NOT import ragas in this cell or before restarting.")

## Cell 0b - Stub the broken VertexAI imports, then import ragas

ragas 0.4.3's `ragas/llms/base.py` has two imports that fail on this image's
langchain_community (0.4.2, officially sunset upstream, `chat_models.vertexai`
already removed):

  line 12: from langchain_community.chat_models.vertexai import ChatVertexAI
  line 13: from langchain_community.llms import VertexAI

These are NOT the same kind of gap and need different fixes:
- Line 12 wants a whole SUBMODULE (`chat_models.vertexai`) that doesn't exist
  -> register a fake module object in sys.modules.
- Line 13 wants a single ATTRIBUTE on the REAL `langchain_community.llms`
  package (which does exist and has other real content we must not break)
  -> import the real package, then patch just the missing attribute onto it.

This project never uses VertexAI (Gemini is called directly via google-genai),
so both stubs raise NotImplementedError if anyone ever tries to actually
instantiate them - they exist only to satisfy import-time references, never
to be used. This workaround must be disclosed in step8.md §4a.

Verification items 1 and 2 (per today's agreed validation plan) happen at
the end of this cell: ragas imports successfully, and llm_factory's real
signature is printed so we know whether provider="google" / client=... (per
step8.md §4's draft) is actually valid before writing any real code against it.

In [ ]:
# Cell 0b: stub broken VertexAI imports, import ragas, inspect llm_factory
import sys, types, inspect

# --- Step A: line 12 needs a whole submodule that doesn't exist ---
CHAT_VERTEX_DOTTED = "langchain_community.chat_models.vertexai"
if CHAT_VERTEX_DOTTED not in sys.modules:
    stub_mod = types.ModuleType(CHAT_VERTEX_DOTTED)

    class ChatVertexAI:
        def __init__(self, *a, **k):
            raise NotImplementedError(
                "ChatVertexAI is a stub inserted to work around a broken "
                "import in ragas 0.4.3 (see step8.md sec 4a). This project "
                "never uses VertexAI; Gemini is called directly via google-genai."
            )

    stub_mod.ChatVertexAI = ChatVertexAI
    sys.modules[CHAT_VERTEX_DOTTED] = stub_mod

    import langchain_community.chat_models as _cm
    _cm.vertexai = stub_mod  # so `from langchain_community.chat_models import vertexai` also resolves
    print("stubbed submodule:", CHAT_VERTEX_DOTTED)
else:
    print("already present, not stubbing:", CHAT_VERTEX_DOTTED)

# --- Step B: line 13 needs one attribute on the REAL langchain_community.llms ---
import langchain_community.llms as _llms
if not hasattr(_llms, "VertexAI"):

    class VertexAI:
        def __init__(self, *a, **k):
            raise NotImplementedError(
                "VertexAI is a stub inserted to work around a broken import "
                "in ragas 0.4.3 (see step8.md sec 4a). This project never "
                "uses VertexAI; Gemini is called directly via google-genai."
            )

    _llms.VertexAI = VertexAI
    print("stubbed attribute: langchain_community.llms.VertexAI")
else:
    print("already present, not stubbing: langchain_community.llms.VertexAI")

# --- Step C: verification item 1 - does ragas import now? ---
import ragas
print("\nVERIFICATION 1 PASSED: ragas imported OK, version:", ragas.__version__)

# --- Step D: verification item 2 - what does llm_factory actually accept? ---
from ragas.llms import llm_factory

sig = inspect.signature(llm_factory)
print("\nVERIFICATION 2 - llm_factory signature:")
print(" ", sig)
print("\nparameters:")
for name, param in sig.parameters.items():
    default = "REQUIRED" if param.default is inspect._empty else repr(param.default)
    print(f"  {name}: default={default}")

## Note: ragas 0.4.3 environment workaround (found 2026-08-01)

`ragas 0.4.3`'s `ragas/llms/base.py` unconditionally imports `ChatVertexAI`
from `langchain_community.chat_models.vertexai` — a path that no longer
exists in this Kaggle image's `langchain_community` (0.4.2, upstream-sunset,
migrated to standalone integration packages). This breaks the entire
`ragas` import chain even though this project never uses VertexAI (Gemini
is called directly via `google-genai`).

Cell 0b stubs the missing submodule with a class that raises
`NotImplementedError` if anyone ever tries to instantiate it — it exists
only to satisfy the import, never to be used. `langchain_community.llms.VertexAI`
(the second import on line 13) turned out to already be present in this
image and did NOT need stubbing.

Confirmed via two zero-quota checks before proceeding:
1. `import ragas` succeeds (version 0.4.3).
2. `inspect.signature(llm_factory)` shows `provider` and `client` params,
   matching the call pattern already planned in `step8.md` §4
   (`llm_factory(..., provider="google", client=client)`).

This workaround is disclosed here and will also be recorded in `step8.md`
§4a once the pilot produces results, per this project's disclosure standard
(same principle as Framework B's self-judge disclosure).

**Addendum (same day): a second missing dependency, and a bug it exposed.**
The first real pilot run (Cell 5, first attempt) failed on all 8 queries
with `ConfigurationError: The 'jsonref' package is required for Gemini
structured output`. Confirmed via elapsed time (~2s/query, far too fast for
a real API round trip) that no Gemini API calls were actually made — this
was a client-side dependency check, not a quota or network failure. Fixed
by `pip install instructor[google-genai]` (Cell 0c), which brings in
`jsonref`.

More importantly, that failed run exposed a bug in the pilot's own error
handling: ragas caught the per-query exception internally and returned
`faithfulness = NaN` instead of raising it, so the original Cell 5 recorded
all 8 queries as `error: None` with a `NaN` score — a silent failure that
violates this project's fail-loud standard (same principle as the
"never silently map unknown enum values" rule in Framework B). Cell 5 was
rewritten to explicitly check for `NaN`/non-finite scores and treat them as
failures (raised and recorded with a populated `error` field), before any
further pilot queries were run. The 8-query checkpoint file from the first
attempt was deleted rather than kept, since every row in it was invalid.

## Cell 0c - Install instructor[google-genai] extras (fixes jsonref error)

Cell 5's failure ("The 'jsonref' package is required for Gemini structured
output") is not a quota or network issue — confirmed by the ~2s elapsed per
query, far too fast for a real API round trip. It's a missing client-side
dependency for ragas's structured-output parsing via `instructor`. The error
message itself names the fix. No API calls in this cell.

In [ ]:
# Cell 0c: install the extras instructor needs for Gemini structured output
import subprocess, sys, importlib

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "instructor[google-genai]"],
    check=True,
)
importlib.invalidate_caches()

import jsonref
print("jsonref installed OK, version:", getattr(jsonref, "__version__", "unknown"))
print("\nNEXT: Run -> Restart Session and Clear Cell Output, then re-run Cell 0, 0b, 1-4.")
print("Cell 5 needs a rewrite before rerunning (see next message) — do not rerun it as-is yet.")

## Cell 1 - Load data, build pmap, load Framework A/B generation records

Three inputs, auto-detected under /kaggle/input by filename (same pattern as
step7a/7b Cell 1, so notebook paths don't need to be hardcoded):

- `qbio_papers.json` (from fairsearch-qbio-processed-raj-jici-yb):
  paper_id -> title/abstract, used to rebuild the exact context format shown
  to the generator.
- `rq2_gen_checkpoint.jsonl` (Framework A, 100 neutral queries)
- `rq2_frameworkB_generation_raw.jsonl` (Framework B, 50 contradictory queries)

Both generation files (from step7_frameworkAB_result_for_Raj) are treated as
optional individually — at least one must be present, per the fail-loud rule,
but the pilot can proceed with just one if the other isn't attached. No API
calls in this cell.

In [ ]:
# Cell 1: load data, auto-detect Kaggle input paths, build pmap
import json
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

def find_file(filename, required=True):
    """Find a file by exact name anywhere under /kaggle/input."""
    hits = list(INPUT_ROOT.rglob(filename))
    if not hits:
        if required:
            raise AssertionError(f"NOT FOUND: {filename} under {INPUT_ROOT}")
        return None
    if len(hits) > 1:
        print(f"WARNING: multiple copies of {filename}: {hits}. Using first.")
    return hits[0]

# --- corpus (title/abstract for rebuilding context) ---
path_papers = find_file("qbio_papers.json")
print("qbio_papers.json ->", path_papers)
papers = json.loads(path_papers.read_text())
pmap = {p["paper_id"]: p for p in papers}
print("corpus papers (pmap):", len(pmap))

# --- Framework A / Framework B generation records ---
path_fwA = find_file("rq2_gen_checkpoint.jsonl", required=False)
path_fwB = find_file("rq2_frameworkB_generation_raw.jsonl", required=False)

assert path_fwA is not None or path_fwB is not None, (
    "Neither Framework A nor Framework B generation file found under "
    "/kaggle/input. Confirm step7_frameworkAB_result_for_Raj is attached."
)
print("Framework A file ->", path_fwA)
print("Framework B file ->", path_fwB)

def load_jsonl(path):
    recs = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                recs.append(json.loads(line))
    return recs

records = []
if path_fwA is not None:
    recs_a = load_jsonl(path_fwA)
    for r in recs_a:
        r["_type"] = "neutral"
    records.extend(recs_a)
if path_fwB is not None:
    recs_b = load_jsonl(path_fwB)
    for r in recs_b:
        r["_type"] = "contradictory"
    records.extend(recs_b)

n_neutral = sum(1 for r in records if r["_type"] == "neutral")
n_contra = sum(1 for r in records if r["_type"] == "contradictory")
print(f"\ntotal generation records loaded: {len(records)} "
      f"(neutral={n_neutral}, contradictory={n_contra})")

# sanity check: every retrieved_paper_id should resolve in the corpus
all_pids = {pid for r in records for pid in r["retrieved_paper_ids"]}
missing = [pid for pid in all_pids if pid not in pmap]
print(f"unique retrieved paper_ids across all records: {len(all_pids)}")
print(f"missing from corpus: {len(missing)}" + (f" -> {missing[:10]}" if missing else " (none)"))

## Cell 2 - Stratified sample for the pilot (seed=42)

Select N_PILOT queries from the 150 loaded records, split roughly evenly
between neutral (Framework A) and contradictory (Framework B) so the pilot
isn't biased toward one query type. No API calls.

In [ ]:
# Cell 2: stratified sample for the pilot (seed=42, N_PILOT queries)
import random

N_PILOT = 8  # change to anything in 5-10 if you want

by_type = {"neutral": [], "contradictory": []}
for r in records:
    by_type[r["_type"]].append(r)

rng = random.Random(42)

present_types = [t for t in ("neutral", "contradictory") if by_type[t]]
per_type_n = N_PILOT // len(present_types)
remainder = N_PILOT - per_type_n * len(present_types)

pilot_records = []
for i, t in enumerate(present_types):
    n = per_type_n + (1 if i < remainder else 0)
    n = min(n, len(by_type[t]))
    pool = sorted(by_type[t], key=lambda r: r["query_id"])
    pilot_records.extend(rng.sample(pool, n))

pilot_records.sort(key=lambda r: r["query_id"])
print(f"pilot sample ({len(pilot_records)} queries, seed=42):")
for r in pilot_records:
    print(f"  {r['query_id']} [{r['_type']}]")

## Cell 3 - Rebuild contexts in the EXACT format shown to the generator

Reconstructs each paper's context chunk using the SAME format as step7a/7b's
build_context() Cell 3: `[i] Title: ...\nAbstract: ...`, in retrieval-rank
order. This matters because Faithfulness checks whether each claim in
answer_text is supported by these contexts — if the format here diverges
from what the model actually saw at generation time, the score would be
measuring something subtly different from what was generated. No API calls.

In [ ]:
# Cell 3: rebuild contexts in the EXACT format shown to the generator
def rebuild_contexts(rec):
    """Return list[str], one per retrieved paper, same [i] Title/Abstract format
    as step7a/7b's build_context()."""
    ctx_list = []
    for i, pid in enumerate(rec["retrieved_paper_ids"], start=1):
        paper = pmap.get(pid)
        if paper is None:
            title, abstract = "(title unavailable)", "(abstract unavailable)"
        else:
            title = (paper.get("title") or "(no title)").strip()
            abstract = (paper.get("abstract") or "(no abstract)").strip()
        ctx_list.append(f"[{i}] Title: {title}\nAbstract: {abstract}")
    return ctx_list

# sanity check on the first pilot record
_test_rec = pilot_records[0]
_test_ctx = rebuild_contexts(_test_rec)
print(f"query_id: {_test_rec['query_id']} [{_test_rec['_type']}]")
print(f"n_contexts: {len(_test_ctx)}")
print(f"answer_text length: {len(_test_rec['answer_text'])} chars")
print("\n--- first context chunk preview ---")
print(_test_ctx[0][:300])

# also confirm all 8 pilot records have exactly 10 contexts each
for r in pilot_records:
    ctx = rebuild_contexts(r)
    assert len(ctx) == 10, f"{r['query_id']}: expected 10 contexts, got {len(ctx)}"
print("\nOK: all 8 pilot records produce exactly 10 contexts each.")

## Cell 4 - Init Gemini client (Kaggle Secrets) + RAGAS evaluator_llm

Builds the google-genai client and wraps it as a ragas evaluator via
llm_factory, per the signature confirmed in Cell 0b
(provider="google", client=client). No API calls yet — this only
constructs Python objects; the first real request happens in Cell 5.

Judge model: gemini-3.1-flash-lite, same as Step 7a/7b generation — a
self-judge design (same model judges its own output), which must be
disclosed identically to Framework B's self-judge limitation.

In [ ]:
# Cell 4: init Gemini client (Kaggle Secrets) + RAGAS evaluator_llm
from kaggle_secrets import UserSecretsClient
from google import genai
from ragas.llms import llm_factory

GEMINI_API_KEY = UserSecretsClient().get_secret("GEMINI_API_KEY")
GEMINI_API_KEY = GEMINI_API_KEY.strip() if GEMINI_API_KEY else GEMINI_API_KEY
assert GEMINI_API_KEY, "key not found; check Secrets label = GEMINI_API_KEY"
print("key loaded from Secrets: OK (length =", len(GEMINI_API_KEY), ")")

JUDGE_MODEL = "gemini-3.1-flash-lite"  # same model as Step 7a/7b, self-judge (disclosed limitation)

client = genai.Client(api_key=GEMINI_API_KEY)
evaluator_llm = llm_factory(JUDGE_MODEL, provider="google", client=client)
print("evaluator_llm constructed:", type(evaluator_llm).__name__)
print("judge model:", JUDGE_MODEL)
print("\nNo API calls made yet. Next cell (Cell 5) is the first one that hits quota.")

## Cell 5 - Per-query Faithfulness run, checkpoint + resume + timing

FIRST CELL THAT CALLS THE GEMINI API. Deliberately does NOT batch all 8
pilot queries into one evaluate() call: per step8.md Sec 4, a single
combined evaluate() would abort the WHOLE batch on an unhandled quota
error. One query = one evaluate() call = one checkpoint line written
immediately after, so a mid-run failure loses at most the current query,
never the queries already done.

Retry: 429/503 get exponential backoff (same pattern as step7a/7b), up to
5 attempts per query. A query that still fails after 5 attempts is recorded
with error != None and faithfulness_score = None, then the loop moves on to
the next query — one failure does not stop the pilot.

In [ ]:
# Cell 5-prep: delete the invalid checkpoint from the first (buggy) attempt
# All 8 rows in it have error=None but faithfulness_score=NaN — not valid data.
import os
CKPT_PATH = "/kaggle/working/step8_pilot_faithfulness.jsonl"
if os.path.exists(CKPT_PATH):
    os.remove(CKPT_PATH)
    print("deleted:", CKPT_PATH)
else:
    print("nothing to delete, file not present")

In [ ]:
# Cell 5: per-query Faithfulness run with checkpoint + resume + timing
# FIXED 2026-08-01: ragas can internally catch a per-query exception and
# return NaN instead of raising it. The original version didn't check for
# this, so all 8 pilot queries were silently recorded as "success" with a
# NaN score. This version explicitly rejects NaN/non-finite scores as
# failures, per this project's fail-loud standard.
import time, os, json, math
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness

CKPT_PATH = "/kaggle/working/step8_pilot_faithfulness.jsonl"
MAX_RETRIES = 5

def load_done_ids(path):
    done = set()
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                line = line.strip()
                if line:
                    try:
                        rec = json.loads(line)
                        # only count as "done" if it was a REAL success —
                        # never resume-skip a row that recorded a failure
                        if rec.get("error") is None and rec.get("faithfulness_score") is not None:
                            done.add(rec["query_id"])
                    except Exception:
                        pass
    return done

def run_faithfulness_one(rec):
    """Build a 1-row Dataset for this query and score Faithfulness only.
    Raises if ragas returns NaN/None instead of a real score."""
    contexts = rebuild_contexts(rec)
    ds = Dataset.from_dict({
        "question": [rec["query_text"]],
        "contexts": [contexts],
        "answer": [rec["answer_text"]],
    })
    result = evaluate(ds, metrics=[faithfulness], llm=evaluator_llm)
    df = result.to_pandas()
    score = df["faithfulness"].iloc[0]
    if score is None or (isinstance(score, float) and math.isnan(score)):
        raise RuntimeError(
            "ragas returned NaN/None for faithfulness — internal job failure "
            "(check the ragas progress-bar output above for 'Exception raised in Job')"
        )
    return float(score)

def run_with_retry(rec):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return run_faithfulness_one(rec)
        except Exception as e:
            msg = str(e)
            transient = ("429" in msg) or ("503" in msg) \
                or ("RESOURCE_EXHAUSTED" in msg) or ("UNAVAILABLE" in msg)
            if transient and attempt < MAX_RETRIES:
                wait = 20 * attempt
                print(f"    {rec['query_id']} transient error (attempt {attempt}); waiting {wait}s")
                time.sleep(wait)
                continue
            raise
    raise RuntimeError("unreachable")

done_ids = load_done_ids(CKPT_PATH)
todo = [r for r in pilot_records if r["query_id"] not in done_ids]
print(f"resume: {len(done_ids)} already done (real successes only) | to run now: {len(todo)}\n")

for rec in todo:
    qid = rec["query_id"]
    t0 = time.time()
    try:
        score = run_with_retry(rec)
        elapsed = time.time() - t0
        row = {
            "query_id": qid, "type": rec["_type"],
            "n_context_papers": len(rec["retrieved_paper_ids"]),
            "faithfulness_score": score,
            "elapsed_sec": round(elapsed, 2), "error": None,
        }
        print(f"  {qid} [{rec['_type']}]: faithfulness={score:.4f} ({elapsed:.1f}s)")
    except Exception as e:
        elapsed = time.time() - t0
        row = {
            "query_id": qid, "type": rec["_type"],
            "n_context_papers": len(rec["retrieved_paper_ids"]),
            "faithfulness_score": None,
            "elapsed_sec": round(elapsed, 2), "error": str(e)[:300],
        }
        print(f"  {qid} FAILED after {elapsed:.1f}s: {str(e)[:100]}")
    with open(CKPT_PATH, "a") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"\ncheckpoint saved -> {CKPT_PATH}")

## Cell 6 - Pilot summary: actual timing, extrapolated to 150 queries

Reads back the checkpoint file and reports the real numbers this pilot was
built to answer (per step8.md Sec 6a item 3): actual call latency and
whether the free tier holds up, before committing to the full 150-query
Tier 1 run. No API calls.

In [ ]:
# Cell 6: pilot summary - actual timing + extrapolation to 150 queries
import json

CKPT_PATH = "/kaggle/working/step8_pilot_faithfulness.jsonl"

all_rows = []
with open(CKPT_PATH) as f:
    for line in f:
        line = line.strip()
        if line:
            all_rows.append(json.loads(line))

ok_rows = [r for r in all_rows if r["error"] is None]
fail_rows = [r for r in all_rows if r["error"] is not None]

print(f"pilot queries attempted : {len(all_rows)}")
print(f"succeeded                : {len(ok_rows)}")
print(f"failed                   : {len(fail_rows)}")

if ok_rows:
    avg_elapsed = sum(r["elapsed_sec"] for r in ok_rows) / len(ok_rows)
    avg_score = sum(r["faithfulness_score"] for r in ok_rows) / len(ok_rows)
    min_score = min(r["faithfulness_score"] for r in ok_rows)
    max_score = max(r["faithfulness_score"] for r in ok_rows)

    print(f"\navg wall time / query    : {avg_elapsed:.2f}s")
    print(f"faithfulness range       : {min_score:.4f} - {max_score:.4f}")
    print(f"avg faithfulness score   : {avg_score:.4f}")

    print(f"\nextrapolated to 150 queries (Tier 1 Faithfulness only):")
    total_sec = avg_elapsed * 150
    print(f"  total wall time        : {total_sec/60:.1f} min ({total_sec/3600:.2f} hr)")
    print(f"  (this is API latency only — add sleep/backoff time if you add any)")

# per-type breakdown, since neutral (Framework A) and contradictory
# (Framework B) queries have different context/answer structure
print("\nby query type:")
for t in ("neutral", "contradictory"):
    rows_t = [r for r in ok_rows if r["type"] == t]
    if rows_t:
        avg_t = sum(r["elapsed_sec"] for r in rows_t) / len(rows_t)
        print(f"  {t:14s}: n={len(rows_t)}, avg elapsed={avg_t:.2f}s")

if fail_rows:
    print("\nfailures (check for quota signal):")
    for r in fail_rows:
        print(f"  {r['query_id']}: {r['error'][:150]}")
else:
    print("\nno failures in this pilot.")

## Cell 7 - Tier 1 Faithfulness, ALL 150 queries (rubric deliverable)

Same per-query checkpoint + resume + NaN-rejection pattern as the fixed
Cell 5, run over all 150 records instead of the 8-query pilot sample. This
is the actual rubric deliverable ("RAGAS Faithfulness scores"), not a
feasibility test. THIS CELL CALLS THE GEMINI API, repeatedly, for a
projected ~30 min based on the Cell 5/6 pilot (11.75s mean per query,
zero 429s observed).

Separate checkpoint file from the pilot's, so the pilot data (8 queries)
and the full run (150 queries) never get mixed up or double-counted.
A small SLEEP between queries is added as an RPM safety margin, since
Faithfulness issues ~2 sub-calls per query and this loop makes far more
total requests than the pilot did.

Safe to stop and rerun this cell at any point: already-completed queries
(real successes only, per the resume fix) are skipped, not redone.

In [ ]:
# Cell 7: Tier 1 Faithfulness, full 150-query run
# Same NaN-rejection logic as the fixed Cell 5. See step8.md sec 4a.5 for
# why the NaN check exists.
import time, os, json, math
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness

CKPT_PATH_FULL = "/kaggle/working/step8_faithfulness_full150.jsonl"
MAX_RETRIES = 5
SLEEP = 3  # small RPM safety margin between queries

def load_done_ids(path):
    done = set()
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                line = line.strip()
                if line:
                    try:
                        rec = json.loads(line)
                        if rec.get("error") is None and rec.get("faithfulness_score") is not None:
                            done.add(rec["query_id"])
                    except Exception:
                        pass
    return done

def run_faithfulness_one(rec):
    contexts = rebuild_contexts(rec)
    ds = Dataset.from_dict({
        "question": [rec["query_text"]],
        "contexts": [contexts],
        "answer": [rec["answer_text"]],
    })
    result = evaluate(ds, metrics=[faithfulness], llm=evaluator_llm)
    df = result.to_pandas()
    score = df["faithfulness"].iloc[0]
    if score is None or (isinstance(score, float) and math.isnan(score)):
        raise RuntimeError(
            "ragas returned NaN/None for faithfulness — internal job failure"
        )
    return float(score)

def run_with_retry(rec):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return run_faithfulness_one(rec)
        except Exception as e:
            msg = str(e)
            transient = ("429" in msg) or ("503" in msg) \
                or ("RESOURCE_EXHAUSTED" in msg) or ("UNAVAILABLE" in msg)
            if transient and attempt < MAX_RETRIES:
                wait = 20 * attempt
                print(f"    {rec['query_id']} transient error (attempt {attempt}); waiting {wait}s")
                time.sleep(wait)
                continue
            raise
    raise RuntimeError("unreachable")

done_ids = load_done_ids(CKPT_PATH_FULL)
todo = [r for r in records if r["query_id"] not in done_ids]
print(f"resume: {len(done_ids)} already done (real successes only) | to run now: {len(todo)}\n")

failed_count = 0
for idx, rec in enumerate(todo, start=1):
    qid = rec["query_id"]
    t0 = time.time()
    try:
        score = run_with_retry(rec)
        elapsed = time.time() - t0
        row = {
            "query_id": qid, "type": rec["_type"],
            "n_context_papers": len(rec["retrieved_paper_ids"]),
            "faithfulness_score": score,
            "elapsed_sec": round(elapsed, 2), "error": None,
        }
    except Exception as e:
        elapsed = time.time() - t0
        failed_count += 1
        row = {
            "query_id": qid, "type": rec["_type"],
            "n_context_papers": len(rec["retrieved_paper_ids"]),
            "faithfulness_score": None,
            "elapsed_sec": round(elapsed, 2), "error": str(e)[:300],
        }
        print(f"  {qid} FAILED after {elapsed:.1f}s: {str(e)[:100]}")
    with open(CKPT_PATH_FULL, "a") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

    if idx % 20 == 0:
        print(f"  progress: {idx}/{len(todo)} attempted, failed so far={failed_count}")
    time.sleep(SLEEP)

# completeness check against the actual 150-query target
done_ids = load_done_ids(CKPT_PATH_FULL)
target_ids = [r["query_id"] for r in records]
missing = [qid for qid in target_ids if qid not in done_ids]
print(f"\nDone. checkpoint has {len(done_ids)}/{len(target_ids)} target queries succeeded.")
if missing:
    print(f"MISSING/FAILED {len(missing)}: {missing[:20]}" + (" ..." if len(missing) > 20 else ""))
    print("-> re-run this cell to retry the missing/failed ones (resume logic skips real successes only)")
else:
    print("All 150 target queries complete.")

In [ ]:
import json
with open('/kaggle/working/step8_faithfulness_full150.jsonl') as f:
    lines = f.readlines()
print(len(lines), 'total lines')
print(lines[0])           # one successful record
for l in lines:
    rec = json.loads(l)
    if rec.get('query_id') == 'q032':
        print(l)
        break

## Cell 8 - Aggregate Tier 1 (Faithfulness) full-150 run

Reads the full150 checkpoint (which may contain duplicate rows per query_id,
since q032/q068 were retried and failures append rather than overwrite).
Dedupes by query_id (prefer a successful record if one exists for that id),
reports succeeded/failed counts and per-type breakdown, and writes
`results/ragas_faithfulness_result.json`. Does not call the API.

In [ ]:
# Cell 8: aggregate Tier 1 (Faithfulness) full-150 run, no API calls
import json
import statistics
from pathlib import Path

CHECKPOINT_PATH = "/kaggle/working/step8_faithfulness_full150.jsonl"
OUTPUT_PATH = "/kaggle/working/ragas_faithfulness_result.json"

# 1. Load all rows, dedupe by query_id (prefer a real success if one exists
#    for that id, since retries append rather than overwrite)
by_qid = {}
total_rows_read = 0
with open(CHECKPOINT_PATH) as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        rec = json.loads(line)
        total_rows_read += 1
        qid = rec["query_id"]
        is_success = rec["error"] is None and rec["faithfulness_score"] is not None
        if qid not in by_qid:
            by_qid[qid] = rec
        else:
            # a later row exists for this qid (a retry) - prefer success
            existing_is_success = (
                by_qid[qid]["error"] is None
                and by_qid[qid]["faithfulness_score"] is not None
            )
            if is_success and not existing_is_success:
                by_qid[qid] = rec
            elif not is_success and not existing_is_success:
                # both failed - keep the later attempt's record (most recent)
                by_qid[qid] = rec

print(f"total rows read from checkpoint: {total_rows_read}")
print(f"unique query_ids after dedup: {len(by_qid)}")

# 2. Verify against the pre-registered 150-query scope
with open("/kaggle/input/datasets/yanbochen928/step7-frameworka-for-raj/queries_all_150.json") as f:
    all_queries = json.load(f)
expected_ids = {q["query_id"] for q in all_queries}
id_to_type = {q["query_id"]: q["type"] for q in all_queries}

missing_from_checkpoint = expected_ids - set(by_qid.keys())
unexpected_in_checkpoint = set(by_qid.keys()) - expected_ids
assert not missing_from_checkpoint, f"missing query_ids: {missing_from_checkpoint}"
assert not unexpected_in_checkpoint, f"unexpected query_ids: {unexpected_in_checkpoint}"
print("all 150 pre-registered query_ids accounted for, no unexpected ids")

# 3. Split succeeded vs failed
succeeded = {qid: r for qid, r in by_qid.items()
             if r["error"] is None and r["faithfulness_score"] is not None}
failed = {qid: r for qid, r in by_qid.items()
          if qid not in succeeded}

print(f"\nsucceeded: {len(succeeded)}/150")
print(f"failed: {len(failed)}/150")
for qid, r in sorted(failed.items()):
    print(f"  {qid} ({r['type']}): error={r['error']!r}, "
          f"elapsed_sec={r['elapsed_sec']}")

# 4. Faithfulness stats, overall and by type
def summarize(records):
    scores = [r["faithfulness_score"] for r in records]
    if not scores:
        return None
    return {
        "n": len(scores),
        "mean": statistics.mean(scores),
        "median": statistics.median(scores),
        "min": min(scores),
        "max": max(scores),
        "stdev": statistics.stdev(scores) if len(scores) > 1 else 0.0,
    }

overall_stats = summarize(list(succeeded.values()))
neutral_stats = summarize([r for r in succeeded.values() if r["type"] == "neutral"])
contradictory_stats = summarize([r for r in succeeded.values() if r["type"] == "contradictory"])

print("\n--- Overall ---")
print(overall_stats)
print("\n--- Neutral (Framework A) ---")
print(neutral_stats)
print("\n--- Contradictory (Framework B) ---")
print(contradictory_stats)

# 5. Build the result JSON, including disclosed limitation for the 2 failures
result = {
    "metric": "faithfulness",
    "judge_model": "gemini-3.1-flash-lite",
    "self_judge_disclosure": (
        "Same model used for both generation (Step 7a/7b) and Faithfulness "
        "judging (Step 8). Self-judge design; see step8.md Section 3 and "
        "the equivalent disclosure for RQ2 Framework B."
    ),
    "query_scope": {
        "target_total": 150,
        "succeeded": len(succeeded),
        "failed": len(failed),
    },
    "failed_queries": [
        {
            "query_id": qid,
            "type": r["type"],
            "error": r["error"],
            "elapsed_sec": r["elapsed_sec"],
            "attempts": 2,
            "disclosed_limitation": (
                "Failed identically on two separate Cell 7 executions "
                "(InstructorRetryException: 'Please return your response "
                "as a function call'), elapsed ~136-138s both times. "
                "Checked and ruled out: answer length (below corpus "
                "median), citation count (below corpus median), citation "
                "formatting (adjacent brackets / two-digit numbers - "
                "common corpus-wide, not distinctive to these two "
                "queries), and non-ASCII characters in retrieved context "
                "(none found). No content-based cause identified; treated "
                "as an unresolved intermittent Gemini/instructor "
                "structured-output failure, not re-attempted a third time."
            ),
        }
        for qid, r in sorted(failed.items())
    ],
    "faithfulness_overall": overall_stats,
    "faithfulness_by_type": {
        "neutral": neutral_stats,
        "contradictory": contradictory_stats,
    },
    "per_query": [
        {
            "query_id": qid,
            "type": r["type"],
            "faithfulness_score": r["faithfulness_score"],
            "n_context_papers": r["n_context_papers"],
            "elapsed_sec": r["elapsed_sec"],
        }
        for qid, r in sorted(succeeded.items())
    ],
}

with open(OUTPUT_PATH, "w") as f:
    json.dump(result, f, indent=2)

print(f"\nwrote {OUTPUT_PATH}")
print(f"succeeded: {len(succeeded)}/150 ({len(succeeded)/150:.1%})")

In [ ]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f == "queries_all_150.json":
            print(os.path.join(root, f))

## Cell 9 - Answer Relevancy pilot (same 8 queries, Tier 2)

Runs ONLY answer_relevancy on the same 8 pilot queries — Faithfulness scores
already exist from Cell 5 and are not recomputed. Separate checkpoint file
so Tier 1 and Tier 2 timings/failures can be compared independently, per
step8.md Sec 2a's tiering (Faithfulness required; Relevancy opportunistic).

Same per-query checkpoint + resume + NaN-rejection pattern as the fixed
Cell 5. THIS CELL CALLS THE GEMINI API.

## Cell 9 (PARKED — do not run)

Blocked on a ragas-internal defect (`embed_query` missing on the
auto-selected GoogleEmbeddings object), not a quota issue. See
step8.md sec 4a.4 for the three reasons this was parked and the
condition for revisiting it as a bounded one-query test.

In [ ]:
# Cell 9: per-query Answer Relevancy pilot, checkpoint + resume + timing (parked)
import time, os, json, math
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import answer_relevancy

CKPT_PATH_REL = "/kaggle/working/step8_pilot_relevancy.jsonl"
MAX_RETRIES = 5

def load_done_ids(path):
    done = set()
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                line = line.strip()
                if line:
                    try:
                        rec = json.loads(line)
                        if rec.get("error") is None and rec.get("answer_relevancy_score") is not None:
                            done.add(rec["query_id"])
                    except Exception:
                        pass
    return done

def run_relevancy_one(rec):
    """Build a 1-row Dataset for this query and score Answer Relevancy only.
    Raises if ragas returns NaN/None instead of a real score."""
    contexts = rebuild_contexts(rec)
    ds = Dataset.from_dict({
        "question": [rec["query_text"]],
        "contexts": [contexts],
        "answer": [rec["answer_text"]],
    })
    result = evaluate(ds, metrics=[answer_relevancy], llm=evaluator_llm)
    df = result.to_pandas()
    score = df["answer_relevancy"].iloc[0]
    if score is None or (isinstance(score, float) and math.isnan(score)):
        raise RuntimeError(
            "ragas returned NaN/None for answer_relevancy — internal job failure"
        )
    return float(score)

def run_with_retry(rec):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return run_relevancy_one(rec)
        except Exception as e:
            msg = str(e)
            transient = ("429" in msg) or ("503" in msg) \
                or ("RESOURCE_EXHAUSTED" in msg) or ("UNAVAILABLE" in msg)
            if transient and attempt < MAX_RETRIES:
                wait = 20 * attempt
                print(f"    {rec['query_id']} transient error (attempt {attempt}); waiting {wait}s")
                time.sleep(wait)
                continue
            raise
    raise RuntimeError("unreachable")

done_ids = load_done_ids(CKPT_PATH_REL)
todo = [r for r in pilot_records if r["query_id"] not in done_ids]
print(f"resume: {len(done_ids)} already done (real successes only) | to run now: {len(todo)}\n")

for rec in todo:
    qid = rec["query_id"]
    t0 = time.time()
    try:
        score = run_with_retry(rec)
        elapsed = time.time() - t0
        row = {
            "query_id": qid, "type": rec["_type"],
            "answer_relevancy_score": score,
            "elapsed_sec": round(elapsed, 2), "error": None,
        }
        print(f"  {qid} [{rec['_type']}]: answer_relevancy={score:.4f} ({elapsed:.1f}s)")
    except Exception as e:
        elapsed = time.time() - t0
        row = {
            "query_id": qid, "type": rec["_type"],
            "answer_relevancy_score": None,
            "elapsed_sec": round(elapsed, 2), "error": str(e)[:300],
        }
        print(f"  {qid} FAILED after {elapsed:.1f}s: {str(e)[:100]}")
    with open(CKPT_PATH_REL, "a") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"\ncheckpoint saved -> {CKPT_PATH_REL}")

## Cell 9b - Inspect embedding_factory signature (no API calls)

Same validation pattern as Cell 0b's llm_factory check: before spending any
quota on Tier 2, confirm what parameters embedding_factory actually accepts
in ragas 0.4.3, rather than guessing provider="google" / client=client by
analogy to llm_factory. If the signature differs, the Cell 9c call below
must be adjusted to match — do not run 9c until this cell's output is read.

In [ ]:
# Cell 9b: inspect embedding_factory signature (zero API calls)
import inspect
from ragas.embeddings import embedding_factory

sig = inspect.signature(embedding_factory)
print("embedding_factory signature:")
print(" ", sig)
print("\nparameters:")
for name, param in sig.parameters.items():
    default = "REQUIRED" if param.default is inspect._empty else repr(param.default)
    print(f"  {name}: default={default}")

In [ ]:
# Cell 9b-2: dig deeper into embedding_factory (still zero API calls)
from ragas.embeddings import embedding_factory
import inspect

print("--- docstring ---")
print(embedding_factory.__doc__)

print("\n--- source ---")
try:
    print(inspect.getsource(embedding_factory))
except Exception as e:
    print("could not get source:", e)

print("\n--- other things available in ragas.embeddings ---")
import ragas.embeddings as emb_mod
print([n for n in dir(emb_mod) if not n.startswith("_")])

In [ ]:
# Cell 9b-3: does GoogleEmbeddings actually have embed_query? (zero API calls)
from ragas.embeddings import GoogleEmbeddings
import inspect

print("--- GoogleEmbeddings methods ---")
print([n for n in dir(GoogleEmbeddings) if not n.startswith("_")])

print("\nhas embed_query:", hasattr(GoogleEmbeddings, "embed_query"))
print("has embed_text:", hasattr(GoogleEmbeddings, "embed_text"))
print("has aembed_query:", hasattr(GoogleEmbeddings, "aembed_query"))

print("\n--- GoogleEmbeddings.__init__ signature ---")
print(inspect.signature(GoogleEmbeddings.__init__))

print("\n--- GoogleEmbeddings source (if available) ---")
try:
    print(inspect.getsource(GoogleEmbeddings))
except Exception as e:
    print("could not get source:", e)

In [ ]:
# Cell 9b-4: check the modern collections API and alternative embedding
# wrappers, before spending any quota (zero API calls)
import inspect

# --- Path A: does ragas.metrics.collections have a modern answer relevancy? ---
print("=== ragas.metrics.collections contents ===")
try:
    import ragas.metrics.collections as coll
    names = [n for n in dir(coll) if not n.startswith("_")]
    print(names)
    # anything relevancy-shaped?
    rel_names = [n for n in names if "elevan" in n.lower()]
    print("\nrelevancy-related names:", rel_names)
    for n in rel_names:
        obj = getattr(coll, n)
        print(f"\n--- {n} ---")
        print("type:", type(obj).__name__)
        try:
            print("init signature:", inspect.signature(obj.__init__))
        except Exception as e:
            print("no init signature:", e)
except Exception as e:
    print("could not import ragas.metrics.collections:", e)

# --- Path C: which embedding classes expose the legacy embed_query? ---
print("\n\n=== embed_query availability across embedding classes ===")
from ragas.embeddings import (
    GoogleEmbeddings,
    HuggingFaceEmbeddings,
    LangchainEmbeddingsWrapper,
    OpenAIEmbeddings,
)
for cls in (GoogleEmbeddings, HuggingFaceEmbeddings,
            LangchainEmbeddingsWrapper, OpenAIEmbeddings):
    print(f"  {cls.__name__:30s} embed_query={hasattr(cls, 'embed_query'):5}  "
          f"embed_text={hasattr(cls, 'embed_text')}")

# --- what interface does the legacy answer_relevancy actually demand? ---
print("\n\n=== legacy answer_relevancy: where is embed_query called? ===")
try:
    from ragas.metrics import answer_relevancy
    src = inspect.getsource(type(answer_relevancy))
    for i, line in enumerate(src.splitlines()):
        if "embed" in line:
            print(f"  line {i}: {line.strip()}")
except Exception as e:
    print("could not inspect:", e)

In [ ]:
# Cell 9b-5: how is a collections-style metric actually invoked? (zero API calls)
import inspect
from ragas.metrics.collections import AnswerRelevancy, BaseMetric

print("=== AnswerRelevancy public methods ===")
print([n for n in dir(AnswerRelevancy) if not n.startswith("_")])

print("\n=== BaseMetric public methods (parent class) ===")
print([n for n in dir(BaseMetric) if not n.startswith("_")])

print("\n=== docstring ===")
print(AnswerRelevancy.__doc__)

# does it still work with the legacy evaluate() dataset-based call,
# or does it need a different single-sample call pattern?
for method_name in ("score", "ascore", "single_turn_ascore", "_ascore", "achat"):
    if hasattr(AnswerRelevancy, method_name):
        m = getattr(AnswerRelevancy, method_name)
        try:
            print(f"\n{method_name} signature: {inspect.signature(m)}")
        except Exception as e:
            print(f"\n{method_name}: could not get signature ({e})")

## Cell 9c - Tier 2 bounded one-query test (modern collections API)

Root cause of the original Cell 9 failure (step8.md sec 4a.1 item 3): the
LEGACY `ragas.metrics.answer_relevancy` calls the old `embed_query`
interface, but `GoogleEmbeddings` only implements the modern
`embed_text`/`embed_texts` interface (confirmed via
`hasattr(GoogleEmbeddings, "embed_query") == False`). The modern
`ragas.metrics.collections.AnswerRelevancy` is typed against
`BaseRagasEmbedding` (the modern base class GoogleEmbeddings belongs to),
so the two should be interface-compatible.

Tests ONE query only (q004 - already succeeded on Faithfulness in Cell 5,
chosen to isolate whether THIS metric/embeddings pairing works, independent
of whether q004's content itself is the problem). Reuses `client` and
`evaluator_llm` from Cell 4 - no need to rebuild them.

Per step8.md sec 4a.4: if this fails with the same or a new unresolved
error, STOP - do not attempt a third fix. Record Tier 2 as a disclosed
limitation instead. THIS CELL CALLS THE GEMINI API (both LLM and embedding
calls), for one query only.

In [ ]:
# Cell 9c: Tier 2 bounded ONE-QUERY test, modern collections API
# THIS CELL CALLS THE GEMINI API (one query only: 1 LLM call region +
# a handful of embedding calls for the generated candidate questions).
from ragas.embeddings import GoogleEmbeddings
from ragas.metrics.collections import AnswerRelevancy

# reuse client + evaluator_llm from Cell 4 - both already "modern" objects
embeddings_obj = GoogleEmbeddings(client=client)

metric = AnswerRelevancy(llm=evaluator_llm, embeddings=embeddings_obj, strictness=3)

# q004: Faithfulness already succeeded for this query (Cell 5), so a
# failure here isolates the metric/embeddings pairing, not the query itself
test_rec = next(r for r in pilot_records if r["query_id"] == "q004")

try:
    result = await metric.ascore(
        user_input=test_rec["query_text"],
        response=test_rec["answer_text"],
    )
    print("query_id:", test_rec["query_id"])
    print("SUCCESS - answer_relevancy score:", result.value)
except Exception as e:
    print("query_id:", test_rec["query_id"])
    print("FAILED:", type(e).__name__, "-", str(e)[:300])
    print("\nPer step8.md sec 4a.4: do not attempt a third fix.")
    print("Record Tier 2 as a disclosed limitation instead.")

In [ ]:
# Cell 9d: is an async client already available? (zero API calls, just inspection)
import inspect

print("does the sync client have an async namespace?")
print("  hasattr(client, 'aio'):", hasattr(client, "aio"))

print("\ndoes llm_factory support building an async-capable LLM directly?")
from ragas.llms import llm_factory
sig = inspect.signature(llm_factory)
print(" ", sig)

print("\ndoes GEMINI_API_KEY-based genai.Client have an async client constructor?")
from google import genai
print("  genai.Client signature:", inspect.signature(genai.Client.__init__))

In [ ]:
# Cell 9e: LAST attempt for Tier 2. One query (q004) only.
# Path 1: use the SYNC score() with the objects already built in Cell 9c.
#   The Cell 9c error said "Use generate() instead", and AnswerRelevancy
#   exposes a sync score() alongside the async ascore(), so the existing
#   sync client may work as-is with no rebuilding.
# Path 2 (only if Path 1 fails): rebuild the LLM against client.aio, the
#   async namespace of the google-genai client (confirmed to exist in
#   Cell 9d). Embeddings deliberately stay on the SYNC client: GoogleEmbeddings
#   .aembed_text wraps the sync call in a thread executor, so an async client
#   there would break it.
from ragas.llms import llm_factory
from ragas.metrics.collections import AnswerRelevancy

print("=== Path 1: sync score() with existing metric ===")
try:
    result = metric.score(
        user_input=test_rec["query_text"],
        response=test_rec["answer_text"],
    )
    print("SUCCESS - answer_relevancy score:", result.value)
    path1_ok = True
except Exception as e:
    print("FAILED:", type(e).__name__, "-", str(e)[:300])
    path1_ok = False

if not path1_ok:
    print("\n=== Path 2: rebuild LLM against client.aio, retry ascore() ===")
    try:
        evaluator_llm_async = llm_factory(JUDGE_MODEL, provider="google", client=client.aio)
        metric_async = AnswerRelevancy(
            llm=evaluator_llm_async,
            embeddings=embeddings_obj,   # stays sync, on purpose
            strictness=3,
        )
        result = await metric_async.ascore(
            user_input=test_rec["query_text"],
            response=test_rec["answer_text"],
        )
        print("SUCCESS - answer_relevancy score:", result.value)
    except Exception as e:
        print("FAILED:", type(e).__name__, "-", str(e)[:300])
        print("\nStop here. Record Tier 2 as a disclosed limitation per step8.md sec 4a.4.")

## Cell 9f - Tier 2 attempt 4: escape the notebook event loop via a thread

Per step8.md sec 4a.8: the notebook's own event loop is what forces the
async ascore() path, which in turn requires an async client that ragas's
instructor adapter then rejects - a closed contradiction inside the main
thread. This is a different mechanism from attempts 1-3 (which varied the
client/embeddings pairing): here the sync score() call runs inside a
worker thread via ThreadPoolExecutor, which has no running event loop of
its own, so ragas's "are we in an async context" check should pass.

Reuses `metric` from Cell 9c (built with the sync client) - no rebuild
needed. Still ONE query only (q004). If this also fails, Tier 2 stays
CLOSED per §4a.8 - this is the last attempt, no fifth workaround.</parameter>

In [ ]:
# Cell 9f: Tier 2 attempt 4 - run sync score() in a worker thread
# THIS CELL CALLS THE GEMINI API (one query only).
from concurrent.futures import ThreadPoolExecutor

def _run_in_thread():
    # inside this worker thread, there is no running event loop,
    # so ragas's sync score() should not detect an "async context"
    return metric.score(
        user_input=test_rec["query_text"],
        response=test_rec["answer_text"],
    )

print("=== Attempt 4: sync score() inside a ThreadPoolExecutor thread ===")
try:
    with ThreadPoolExecutor(max_workers=1) as executor:
        future = executor.submit(_run_in_thread)
        result = future.result(timeout=60)
    print("SUCCESS - answer_relevancy score:", result.value)
except Exception as e:
    print("FAILED:", type(e).__name__, "-", str(e)[:300])
    print("\nTier 2 remains CLOSED (step8.md sec 4a.8). No further attempts.")

## Cell 10 - Visualize Tier 1 (Faithfulness) results

Reads the aggregated `ragas_faithfulness_result.json` written by Cell 8 and
plots mean faithfulness by query type (overall / neutral / contradictory).
Also lists the 2 unresolved failures (q032, q068) for reference. No API
calls; reads only the local aggregation file.

In [ ]:
# Cell 10: visualize Tier 1 (Faithfulness) results, no API calls
import json
import matplotlib.pyplot as plt

with open("/kaggle/working/ragas_faithfulness_result.json") as f:
    result = json.load(f)

overall = result["faithfulness_overall"]
neutral = result["faithfulness_by_type"]["neutral"]
contradictory = result["faithfulness_by_type"]["contradictory"]

labels = [
    f"Overall (n={overall['n']})",
    f"Neutral (n={neutral['n']})",
    f"Contradictory (n={contradictory['n']})",
]
means = [overall["mean"], neutral["mean"], contradictory["mean"]]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(labels, means, color="#2a78d6", width=0.5)
ax.set_ylim(0.9, 1.0)
ax.set_ylabel("Mean faithfulness score")
n_succeeded = result["query_scope"]["succeeded"]
n_total = n_succeeded + result["query_scope"]["failed"]
ax.set_title(f"Tier 1 Faithfulness: {n_succeeded}/{n_total} queries succeeded")
for bar, val in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.002, f"{val:.4f}", ha="center", fontsize=10)
ax.grid(axis="y", linestyle="--", alpha=0.3)
plt.tight_layout()
plt.savefig("/kaggle/working/step8_faithfulness_chart.png", dpi=150)
plt.show()

print(f"\nfailed queries ({len(result['failed_queries'])}):")
for fq in result["failed_queries"]:
    print(f"  {fq['query_id']} ({fq['type']}): {fq['error']}")

## Cell 9 summary - Tier 2 (Answer Relevancy) investigation, CLOSED

Four attempts were made to get Answer Relevancy working on this stack,
each responding to the specific error raised by the previous one. Full
trail recorded in `step8.md` sec 4a.8.

| # | Approach | Result |
|---|---|---|
| 0 | Legacy `ragas.metrics.answer_relevancy` + `GoogleEmbeddings` | `AttributeError: 'GoogleEmbeddings' object has no attribute 'embed_query'` |
| 1 | Modern `collections.AnswerRelevancy` + `GoogleEmbeddings`, sync client, `ascore()` | embeddings layer passed; `TypeError: Cannot use agenerate() with a synchronous client` |
| 2 | Same objects, sync `score()` instead | `RuntimeError: Cannot call sync score() from an async context` |
| 3 | LLM rebuilt on `client.aio` (async), `ascore()` | `ValueError: Client must be an instance of google.genai.Client. Got: AsyncClient` |
| 4 | Sync `score()` run inside a `ThreadPoolExecutor` worker thread | Escaped the event-loop check, but hit the same `TypeError` as attempt 1 - the LLM object itself still required an async-capable client regardless of thread context |

**Conclusion: Tier 2 is closed as infeasible on ragas 0.4.3 + google-genai
in this notebook environment.** Attempt 0's interface mismatch (legacy vs
modern embeddings API) was genuinely solved. What remains is a closed
contradiction between ragas's async-only `ascore()` path and its
instructor adapter's rejection of async clients - confirmed by attempt 4,
which proved the deadlock is about client typing, not the notebook's event
loop. Not a quota issue, not a configuration error, and no fifth
workaround is planned. Answer Relevancy is not in the rubric (`step8.md`
sec 2a), so this closes an optional metric, not a required one.